# Yambda 06 · Is the iALS tuning real? (edges + significance)

Two problems with `05`:
1. **Three of the four winners sit on the edge of their grid**: reg = 5 (largest tried), alpha = 30
   (largest), factors = 256 (largest). The optimum may lie further out.
2. **Many gaps are tiny** (half-life 1/3/5/7 days span 0.0294–0.0303). We don't know which
   differences are real and which are noise from *which users happened to be evaluated*.

This notebook (a) extends the grid past the edges and (b) tests the key comparisons with a paired
bootstrap over users. ~20 min.

In [1]:
import json, time
import pandas as pd
from musicrec import config
from musicrec.preprocess import Split
from musicrec.evaluation import evaluate, paired_bootstrap
from musicrec.models.baselines import MostPopular
from musicrec.models.als import IALS

pd.set_option("display.float_format", "{:,.4f}".format)
split = Split.load(config.YAMBDA_SPLIT_VAL_DIR)
OUT = config.PROJECT_ROOT / "reports" / "yambda"

runs, params = {}, {}   # name -> evaluate(...) result incl. per-user scores / hyperparameters
def run(model):
    t = time.time()
    model.fit(split)
    runs[model.name] = res = evaluate(model, split, return_per_user=True)
    params[model.name] = model.params()
    print(f"{model.name:<55} recall@10={res['recall@10']:.4f}  ({time.time() - t:.0f}s)")
    return res["recall@10"]

pop = MostPopular(by="listeners", recent_days=7).fit(split)
runs[pop.name] = evaluate(pop, split, return_per_user=True)

## E1 · reg × alpha, beyond the old edges (f = 128 to keep it fast)

Searched jointly because the two interact: a larger alpha makes the data term of each
least-squares problem larger, so the best regularization tends to grow with it.

In [2]:
for reg in [5.0, 20.0, 50.0]:
    for alpha in [30, 100]:
        run(IALS(factors=128, regularization=reg, alpha=alpha, half_life_days=3))
best128 = max((n for n in runs if n.startswith("iALS(f=128")), key=lambda n: runs[n]["recall@10"])
print("best at f=128:", best128)

iALS(f=128, reg=5.0, alpha=30, half-life 3d)            recall@10=0.0316  (67s)
iALS(f=128, reg=5.0, alpha=100, half-life 3d)           recall@10=0.0305  (68s)
iALS(f=128, reg=20.0, alpha=30, half-life 3d)           recall@10=0.0316  (68s)
iALS(f=128, reg=20.0, alpha=100, half-life 3d)          recall@10=0.0316  (68s)
iALS(f=128, reg=50.0, alpha=30, half-life 3d)           recall@10=0.0324  (65s)
iALS(f=128, reg=50.0, alpha=100, half-life 3d)          recall@10=0.0319  (67s)
best at f=128: iALS(f=128, reg=50.0, alpha=30, half-life 3d)


## E2 · capacity beyond 256, and half-life re-checked at the new settings

In [3]:
p = params[best128]
for f in [256, 512]:
    run(IALS(**{**p, "factors": f}))
best = max((n for n in runs if n.startswith("iALS")), key=lambda n: runs[n]["recall@10"])
P = params[best]
run(IALS(**{**P, "half_life_days": 7}))          # does 3d vs 7d still matter here?
print("best overall:", best)

iALS(f=256, reg=50.0, alpha=30, half-life 3d)           recall@10=0.0333  (99s)
iALS(f=512, reg=50.0, alpha=30, half-life 3d)           recall@10=0.0346  (194s)
iALS(f=512, reg=50.0, alpha=30, half-life 7d)           recall@10=0.0313  (197s)
best overall: iALS(f=512, reg=50.0, alpha=30, half-life 3d)


## Which differences are real?

Each row: is model **b** better than model **a**? `ci_low`/`ci_high` = 95% interval of the
mean per-user difference. If the interval contains 0, we can't tell them apart.

In [4]:
old_best = IALS(factors=128, regularization=0.05, alpha=10, half_life_days=7)   # winner of notebook 04
run(old_best)
hl7 = IALS(**{**P, "half_life_days": 7}).name
f128 = IALS(**{**P, "factors": 128}).name

comparisons = [
    ("best vs popularity",          pop.name,      best),
    ("best vs notebook-04 winner",  old_best.name, best),
    ("half-life 3d vs 7d",          hl7,           best),
    ("bigger model vs f=128",       f128,          best),
]
rows = []
for label, a, b_ in comparisons:
    if a == b_:
        continue
    r = paired_bootstrap(runs[a]["per_user"], runs[b_]["per_user"])
    rows.append({"comparison": label, "a": a, "b": b_, **r})
sig = pd.DataFrame(rows).set_index("comparison")
sig[["a", "b", "diff", "rel_diff", "ci_low", "ci_high", "p(b<=a)"]]

iALS(f=128, reg=0.05, alpha=10, half-life 7d)           recall@10=0.0294  (66s)


,a,b,diff,rel_diff,ci_low,ci_high,p(b<=a)
comparison,,,,,,,
best vs popularity,0.0235,0.0346,0.0111,0.4725,0.0076,0.0145,0.0000
best vs notebook-04 winner,0.0294,0.0346,0.0052,0.1754,0.0031,0.0072,0.0000
half-life 3d vs 7d,0.0313,0.0346,0.0033,0.1049,0.0012,0.0053,0.0010
bigger model vs f=128,0.0324,0.0346,0.0022,0.0664,0.0001,0.0041,0.0170


In [5]:
table = pd.DataFrame({n: {k: v for k, v in r.items() if k != "per_user"} for n, r in runs.items()}).T
table = table[["recall@10", "ndcg@10", "hit_rate@10", "recall@50", "coverage@10", "avg_pop@10"]].sort_values("recall@10", ascending=False)
table.to_csv(OUT / "ials_refine.csv")
table

,recall@10,ndcg@10,hit_rate@10,recall@50,coverage@10,avg_pop@10
"iALS(f=512, reg=50.0, alpha=30, half-life 3d)",0.0346,0.0322,0.1671,0.0686,0.0549,832.0774
"iALS(f=256, reg=50.0, alpha=30, half-life 3d)",0.0333,0.0307,0.1661,0.0683,0.0458,912.3873
"iALS(f=128, reg=50.0, alpha=30, half-life 3d)",0.0324,0.0305,0.1613,0.0674,0.0370,991.5701
"iALS(f=128, reg=50.0, alpha=100, half-life 3d)",0.0319,0.0292,0.1534,0.0657,0.0474,922.6851
"iALS(f=128, reg=5.0, alpha=30, half-life 3d)",0.0316,0.0295,0.1580,0.0662,0.0410,957.1736
"iALS(f=128, reg=20.0, alpha=30, half-life 3d)",0.0316,0.0294,0.1571,0.0670,0.0398,965.8690
"iALS(f=128, reg=20.0, alpha=100, half-life 3d)",0.0316,0.0288,0.1539,0.0652,0.0485,917.1171
"iALS(f=512, reg=50.0, alpha=30, half-life 7d)",0.0313,0.0286,0.1550,0.0661,0.0621,831.3292
"iALS(f=128, reg=5.0, alpha=100, half-life 3d)",0.0305,0.0282,0.1522,0.0642,0.0502,909.6047
"iALS(f=128, reg=0.05, alpha=10, half-life 7d)",0.0294,0.0271,0.1511,0.0618,0.0397,"1,018.7152"


In [6]:
# Record the winner only if it is significantly better than the notebook-05 choice;
# otherwise keep the cheaper/simpler model. Printed either way, decided together.
prev = json.loads((OUT / "best_ials.json").read_text())
print("notebook 05 choice:", prev)
print("this notebook's best:", best, f"recall@10={runs[best]['recall@10']:.4f}")

notebook 05 choice: {'params': {'factors': 256, 'regularization': 5.0, 'alpha': 30, 'iterations': 15, 'weighting': 'log', 'half_life_days': 3}, 'val_recall@10': 0.03308292378827779}
this notebook's best: iALS(f=512, reg=50.0, alpha=30, half-life 3d) recall@10=0.0346
